In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import display  # noqa: E402
from scipy.stats import pearsonr, wilcoxon  # noqa: E402

from src.analysis import iva_quality  # noqa: E402
from src.analysis.isc import FREQUENCY_BANDS  # noqa: E402
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExperimentNames,
    IvaVariants,
    MusicTypeVariants,
    SpectrumTypeVariants,
)
from src.io.iva_store import list_iva_results, load_iva_components  # noqa: E402
from src.visualization.iva_condition_plots import (  # noqa: E402
    plot_condition_mean_tf_maps,
    plot_condition_mean_topomaps,
    plot_participant_condition_tf_maps,
    plot_participant_condition_topomaps,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# Analysis of the Stored IVA Components — Both Conditions on the Subject Axis

Reads the components that
[`scripts/run_iva_condition_comparison.py`](../../scripts/run_iva_condition_comparison.py)
wrote with `--store_components` and analyses them. **Nothing is decomposed here.**
The decomposition is the expensive part — a per-recording PCA plus `iva_g` over a
tensor of tens of gigabytes, which belongs on a node
([`run_condition_comparison.pbs`](../../jobs/metacentrum/06-iva-condition-comparison/run_condition_comparison.pbs))
— and its figures are lossy summaries of it: a condition-mean grid cannot be re-split
per participant, and a plotted TF map cannot be correlated against anything. Keeping
the recovered components on disk turns every question below into a load.

**The layout this notebook reads.** Placebo and Psilocybin were pooled on the
**subject** axis
([`ConditionVariants.JOINED`](../../src/definitions/fields.py)), so:

```
row 0 .. P-1    Placebo      recordings      \
row P .. 2P-1   Psilocybin   recordings      /  one IVA dataset each
```

Every *recording* is one dataset, so **a participant occupies two rows** and their two
recordings got their own mixing matrices. Two consequences shape everything below:

- The channel topographies are **per recording**, so a topography contrast between the
  conditions is meaningful — that is what this variant buys.
- A bare participant label addresses **two** rows, so the row bookkeeping is not
  optional. `results.row(participant, condition)` resolves it and refuses to guess.

The sibling notebook
[`iva_component_analysis_joined_tracks.ipynb`](iva_component_analysis_joined_tracks.ipynb)
does the same for the **time**-concatenated join, where each participant is one row
carrying both tracks and the topography is shared.

**What is on disk.** From
[`src/io/iva_store.py`](../../src/io/iva_store.py): the sign-aligned per-recording TF
maps `(recordings, components, freqs, times)`, the channel topographies
`(recordings, components, channels)`, the frequency / time / channel axes, the
participant and condition of every row, the sign-alignment diagnostics, and the
per-condition stimulus onsets.

**Prerequisite.** A stored entry for the settings in the config cell. If the load
fails, run:

```bash
python scripts/run_iva_condition_comparison.py \
    --experiment assr --n_pca 10 --reuse_wavelets --store_components
```

## Configuration

In [ ]:
# ── Which stored decomposition to analyse ──────────────────────
# These five values are exactly what the store filename encodes, so they have to match
# the run that wrote it. Step 1's listing prints every entry if the settings are not
# remembered.
EXPERIMENT_NAME = ExperimentNames.ASSR
# The subject-axis join: one row per (participant, condition).
CONDITION = ConditionVariants.JOINED
VARIANT = IvaVariants.CHANNEL_JOINED
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPE = MusicTypeVariants.ASSR
else:
    MUSIC_TYPE = MusicTypeVariants.CLASSICAL
# The band the run was restricted to, or None for a broadband run.
BAND: str | None = None
# The run's --n_pca, which is also its component count.
N_COMPONENTS_PCA = 5

# Processed-data root the store is resolved against. None = the project's
# data/processed, which is where the CLI writes by default.
STORE_ROOT: Path | None = None

# What resolved the per-recording sign in the run, named on every figure so a reader
# knows which orientation convention they are looking at. Matches the CLI script.
ALIGNMENT_NOTE = "PC1 of the per-recording TF maps (strongest bin positive)"

# ── Which components to draw ──────────────────────────────────
# None = every stored component. A list is 0-based, matching the IC <k+1> labels.
COMPONENTS_TO_PLOT: list[int] | None = None

# ── Figure output ─────────────────────────────────────────────
SAVE_PLOTS = True
# Per-participant grids are one figure PER COMPONENT and there can be many of them,
# so they are opt-in rather than part of a routine pass through the notebook.
WRITE_PARTICIPANT_GRIDS = False

# ── Reference lines on the TF panels ──────────────────────────
# The ASSR is continuous 40 Hz stimulation, so the stimulation frequency is the row
# worth locating on every map.
TF_FREQ_MARKS: list[float] = [iva_quality.ASSR_FREQ]
MARK_STIMULUS_ONSETS_ON_TF = True

# ── The window the paired contrast summarises ─────────────────
# A TF map is (frequency x time) per component; a paired test needs ONE number per
# (participant, component). This window is that reduction: the mean of the stored,
# sign-aligned source over a frequency band and a time span. Because the sources are
# z-scored along time before the decomposition, a positive mean reads as "more power
# in this band than this recording's own average", not as absolute power.
#
# Default: a narrow band around the ASSR stimulation frequency, whole time axis.
# Set CONTRAST_BAND to a FrequencyBandNames value instead to use a standard band.
CONTRAST_FREQ_RANGE: tuple[float, float] = (
    iva_quality.ASSR_FREQ - 2.0,
    iva_quality.ASSR_FREQ + 2.0,
)
CONTRAST_BAND: str | None = None  # e.g. "alpha"; overrides CONTRAST_FREQ_RANGE
CONTRAST_TIME_RANGE: tuple[float, float] | None = None  # None = the whole time axis

# ── Stimulus-locked epoch ─────────────────────────────────────
# Taken from src.definitions.constants.AssrEpoch so this notebook cuts the SAME epoch
# as every other onset-locked ASSR analysis: a short pre-onset baseline, then the
# stimulus plus an equally long post-stimulus interval. iva_quality.onset_window caps
# the post-onset span by the shortest inter-onset gap, so an epoch can never reach the
# next stimulus.
MIN_ONSETS_FOR_EPOCH_AVERAGE = 5

# ── An alignment weaker than this is called out ───────────────
# PC1's share of the ensemble power, from the stored sign alignment. Below this there
# is no single dominant shared map, so the group mean of that component — and any
# contrast built on it — is weak evidence.
WEAK_ALIGNMENT_EVR = 0.5

# ── Plots directory ───────────────────────────────────────────
# Canonical notebook layout. The analysis type carries a "_stored" suffix so these
# figures never overwrite the ones the decomposition notebook writes from its own
# in-memory results — same components, different provenance, worth keeping apart.
SPECTRUM_DIR = (
    SpectrumTypeVariants.BROADBAND.value
    if BAND is None
    else SpectrumTypeVariants.BANDS.value
)
ANALYSIS_TYPE = f"{VARIANT.value}_stored"
PLOTS_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06-iva-condition-comparison"
    / "plots"
    / EXPERIMENT_NAME.value
    / SPECTRUM_DIR
    / ANALYSIS_TYPE
    / f"pca_{N_COMPONENTS_PCA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
# Band runs share the directory with nothing, but keep the prefix so a figure filename
# still says which spectrum it came from.
PLOT_PREFIX = "" if BAND is None else f"{BAND}_"

print(f"Experiment      : {EXPERIMENT_NAME.value}")
print(f"Stored run      : {VARIANT.value} / {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Spectrum        : {BAND or SpectrumTypeVariants.BROADBAND.value}")
print(f"Components      : {N_COMPONENTS_PCA}  (the run's --n_pca)")
print(f"Store root      : {STORE_ROOT or ProjectPaths.PROCESSED_DATA_DIR}")
print(f"Plots directory : {PLOTS_DIR}  (saving: {SAVE_PLOTS})")

## Data Loading — read the stored decomposition

Two calls. `list_iva_results` enumerates what the experiment has on disk, which is the
quickest way to see which runs exist and what settings they used — the filename carries
the variant, the music type, the spectrum and `n_pca`. `load_iva_components` then reads
one entry, either by path or by the run descriptor as here.

An entry is one decomposition. A `--n_pca` or band sweep leaves one file per setting,
so nothing below can silently mix two runs.

In [ ]:
available = list_iva_results(EXPERIMENT_NAME, processed_data_dir=STORE_ROOT)
print(f"Stored IVA results for {EXPERIMENT_NAME.value} ({len(available)}):")
for entry in available:
    print(f"  {entry.parent.name:<14} {entry.name}")
if not available:
    print(
        "  (nothing stored yet — run the CLI with --store_components first:\n"
        f"   python scripts/run_iva_condition_comparison.py --experiment "
        f"{EXPERIMENT_NAME.value} --n_pca {N_COMPONENTS_PCA} --reuse_wavelets "
        "--store_components)"
    )

results = load_iva_components(
    experiment=EXPERIMENT_NAME,
    condition=CONDITION,
    variant=VARIANT,
    music_type=MUSIC_TYPE,
    band=BAND,
    n_pca=N_COMPONENTS_PCA,
    processed_data_dir=STORE_ROOT,
)
print(f"\nLoaded {results.path}")

## Dataset Selection — unpack the arrays and the row bookkeeping

One stored entry, so this cell just gives its contents the names the rest of the
notebook uses, and pulls out the three things that turn arrays back into an analysis:
the **row bookkeeping** (participant and condition per row), the **axes** (frequency,
time, channels), and the **onsets**.

`results.topo_info()` rebuilds an MNE `Info` from the stored channel names and applies
the project montage — that is why the names were stored, since a channel pattern is
only a topography once its values sit on a scalp.

In [ ]:
LABEL = results.label  # the canonical "Joined_<MusicType>" product name

tf_maps = results.tf_maps  # (S, K, F, T) per-recording component TF maps
channel_patterns = results.channel_patterns  # (S, K, C) per-recording topographies
freqs = results.freqs  # (F,) Hz
times = results.times  # (T,) s
sfreq = results.sfreq

n_subjects, n_components, n_freqs, n_times = tf_maps.shape
n_channels = results.n_channels

# The row bookkeeping. Every array above is indexed (recording, component, ...), so
# these are what turn a row index back into a person under a condition — and, here,
# what keeps a participant's two rows apart.
subject_participants = list(results.participants)
subject_conditions = list(results.subject_conditions)
# Condition order as the run laid the subject axis out: one whole block each.
condition_rows = list(dict.fromkeys(subject_conditions))
participants = sorted(set(subject_participants))
# Participants contributing every condition — the ones a paired read-out can use.
paired_participants = [
    p for p in participants if all(results.rows(p, c) for c in condition_rows)
]

comp_indices = (
    list(range(n_components))
    if COMPONENTS_TO_PLOT is None
    else list(COMPONENTS_TO_PLOT)
)
out_of_range = [k + 1 for k in comp_indices if not 0 <= k < n_components]
if out_of_range:
    raise ValueError(
        f"COMPONENTS_TO_PLOT names IC {out_of_range}, outside 1..{n_components}."
    )

# Stimulus onsets, on the one shared time axis (the conditions were aligned together).
# ``None`` for an experiment without stimulus annotations.
onsets_by_condition = {c: results.stimulus_onsets(c) for c in condition_rows}
reference_onsets = onsets_by_condition.get(condition_rows[0])
onset_times = (
    reference_onsets / sfreq
    if MARK_STIMULUS_ONSETS_ON_TF and reference_onsets is not None
    else np.array([])
)

# The topomap layout, rebuilt from the stored channel names.
topo_info = results.topo_info()

print(f"Product     : {LABEL}")
print(f"TF maps     : {tf_maps.shape}  (recordings x components x freqs x times)")
print(f"Topographies: {channel_patterns.shape}  (recordings x components x channels)")
print(f"Time axis   : {times[-1]:.1f} s @ {sfreq} Hz")
print(f"Freq axis   : {freqs[0]:.1f}-{freqs[-1]:.1f} Hz ({n_freqs} bins)")
print(f"Conditions  : {condition_rows}")
print(
    f"Participants: {len(participants)} ({len(paired_participants)} with every "
    "condition)"
)
print(f"Components  : showing {len(comp_indices)} of {n_components}")
print(f"Onsets      : {0 if reference_onsets is None else len(reference_onsets)}")

# ── Small helpers used by several steps below ─────────────────


def _grid(n_panels: int, width: float = 3.6, height: float = 2.9):
    """A subplot grid wide enough for *n_panels*, at most 5 columns."""
    ncols = min(5, max(1, n_panels))
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(width * ncols, height * nrows), squeeze=False
    )
    for j in range(n_panels, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")
    return fig, axes, ncols


def _save(fig, name: str) -> None:
    """Write *fig* into PLOTS_DIR under the band prefix, when saving is on."""
    if not SAVE_PLOTS:
        return
    path = PLOTS_DIR / f"{PLOT_PREFIX}{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path}")


def _benjamini_hochberg(pvals) -> np.ndarray:
    """BH step-up adjusted p-values; ``NaN`` entries pass through untouched.

    Written out rather than imported so the notebook has no statsmodels dependency.
    """
    p = np.asarray(pvals, dtype=float)
    adjusted = np.full(p.shape, np.nan)
    finite = ~np.isnan(p)
    if not finite.any():
        return adjusted
    vals = p[finite]
    order = np.argsort(vals)
    n = vals.size
    stepped = vals[order] * n / np.arange(1, n + 1)
    # Enforce monotonicity from the largest p downwards.
    stepped = np.minimum.accumulate(stepped[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(stepped, 0.0, 1.0)
    adjusted[finite] = out
    return adjusted


def _contrast_masks():
    """``(freq_mask, time_mask, description)`` for the paired-contrast window."""
    if CONTRAST_BAND is not None:
        low, high = FREQUENCY_BANDS[CONTRAST_BAND]
        band_name = CONTRAST_BAND
    else:
        low, high = CONTRAST_FREQ_RANGE
        band_name = f"{low:.1f}-{high:.1f} Hz"
    freq_mask = (freqs >= low) & (freqs <= high)
    if not freq_mask.any():
        raise ValueError(
            f"No stored frequency falls in [{low}, {high}] Hz; the stored grid is "
            f"{freqs[0]:.1f}-{freqs[-1]:.1f} Hz. Adjust CONTRAST_FREQ_RANGE / "
            "CONTRAST_BAND, or load a run whose band covers it."
        )
    return freq_mask, band_name


def _time_mask(axis_times: np.ndarray) -> np.ndarray:
    """Boolean mask over *axis_times* for CONTRAST_TIME_RANGE (None = everything)."""
    if CONTRAST_TIME_RANGE is None:
        return np.ones(axis_times.size, dtype=bool)
    start, stop = CONTRAST_TIME_RANGE
    mask = (axis_times >= start) & (axis_times <= stop)
    if not mask.any():
        raise ValueError(
            f"No stored sample falls in [{start}, {stop}] s; the axis spans "
            f"0-{axis_times[-1]:.1f} s. Adjust CONTRAST_TIME_RANGE."
        )
    return mask

---
## Step 1 — What is in the file

Before any analysis, the two things that make a stored decomposition trustworthy:
the **row bookkeeping** (which participant, under which condition, each row of every
array is) and the **axes** (the frequency, time and channel grids the arrays are on).
Both come out of the file; neither is a convention this notebook has to remember.

`participant_frame()` is the mapping in table form — the thing that lets a row index
be turned back into a person.

In [ ]:
print(f"Variant           : {results.variant.value}")
print(f"Product           : {LABEL}   (store condition {results.condition.value})")
print(f"Spectrum          : {results.band or SpectrumTypeVariants.BROADBAND.value}")
print(f"Components        : {n_components}")
print(
    f"Recordings        : {n_subjects} = {len(participants)} participant(s) x "
    f"{len(condition_rows)} condition(s)"
)
print(f"Time axis         : {n_times} samples, {times[-1]:.1f} s @ {sfreq} Hz")
print(f"Frequency axis    : {n_freqs} bins, {freqs[0]:.1f}-{freqs[-1]:.1f} Hz")
print(f"Channels          : {n_channels}")
print(f"Stored arrays     : {sorted(results.arrays)}")
print(f"Stored diagnostics: {sorted(results.extras)}")
print(f"Onsets stored for : {results.onset_conditions or 'none'}")

# The writer validates these; re-check on read so a hand-edited or truncated file
# cannot quietly mis-address rows.
if not len(subject_participants) == n_subjects == len(subject_conditions):
    raise ValueError("Row bookkeeping does not match the arrays.")

frame = results.participant_frame()
conditions_per_participant = frame.groupby("participant")["condition"].nunique()
print(
    f"\nConditions per participant: min {conditions_per_participant.min()}, "
    f"max {conditions_per_participant.max()}"
)
if conditions_per_participant.nunique() != 1:
    print(
        "  NOTE: unbalanced — a participant is missing a condition, so the paired "
        "steps below will use the balanced subset only."
    )
display(frame)